# Reproducible Visualizations for the GTSRB Paper

Notebook này tạo toàn bộ hình sử dụng trong báo cáo từ dữ liệu GTSRB và đúng pipeline HOG--LinearSVC của nhóm. Không có số liệu hoặc hình minh họa được nhập thủ công. Mỗi hình được lưu ở cả định dạng PNG và PDF trong thư mục `/content/paper_figures`.

## 1. Cài đặt thư viện

Colab thường đã có các thư viện này. Cell dưới đây bảo đảm môi trường có đủ công cụ đọc ảnh, trích xuất HOG, huấn luyện LinearSVC và vẽ biểu đồ.

In [ ]:
!pip install -q scikit-image scikit-learn seaborn joblib tqdm

## 2. Tải và giải nén GTSRB

Ba tệp được tải trực tiếp từ kho công khai chứa training images, official test images và official test ground truth. Cell có kiểm tra tệp tồn tại nên có thể chạy lại mà không tải trùng.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

DATA_ROOT = Path('/content/data')
ZIP_ROOT = DATA_ROOT / 'archives'
FIGURE_DIR = Path('/content/paper_figures')
CACHE_DIR = Path('/content/cache')

for directory in (DATA_ROOT, ZIP_ROOT, FIGURE_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370'
ARCHIVES = [
    'GTSRB_Final_Training_Images.zip',
    'GTSRB_Final_Test_Images.zip',
    'GTSRB_Final_Test_GT.zip',
]

for archive_name in ARCHIVES:
    archive_path = ZIP_ROOT / archive_name
    if not archive_path.exists():
        print(f'Downloading {archive_name} ...')
        urlretrieve(f'{BASE_URL}/{archive_name}', archive_path)
    else:
        print(f'Already downloaded: {archive_path}')

    marker = CACHE_DIR / f'{archive_name}.extracted'
    if not marker.exists():
        print(f'Extracting {archive_name} ...')
        with ZipFile(archive_path, 'r') as zip_file:
            zip_file.extractall(DATA_ROOT)
        marker.touch()
    else:
        print(f'Already extracted: {archive_name}')

print('Dataset preparation completed.')

## 3. Import, cấu hình và tên 43 lớp

Cell này cố định random seed, cấu hình hình phù hợp cho bài báo, và khai báo tên lớp GTSRB để nhãn trong các hình có thể đọc được.

In [ ]:
import json
import os
import textwrap
import time

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from matplotlib.patches import Rectangle
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from skimage import exposure
from skimage.feature import hog
from tqdm.auto import tqdm

RANDOM_STATE = 42
IMG_SIZE = (32, 32)
np.random.seed(RANDOM_STATE)

CLASS_NAMES = {
    0: 'Speed limit (20 km/h)',
    1: 'Speed limit (30 km/h)',
    2: 'Speed limit (50 km/h)',
    3: 'Speed limit (60 km/h)',
    4: 'Speed limit (70 km/h)',
    5: 'Speed limit (80 km/h)',
    6: 'End speed limit (80 km/h)',
    7: 'Speed limit (100 km/h)',
    8: 'Speed limit (120 km/h)',
    9: 'No passing',
    10: 'No passing (>3.5 t)',
    11: 'Right-of-way intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Vehicles >3.5 t prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve left',
    20: 'Dangerous curve right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End speed/passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End no passing',
    42: 'End no passing (>3.5 t)',
}

sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

def save_figure(fig, stem):
    png_path = FIGURE_DIR / f'{stem}.png'
    pdf_path = FIGURE_DIR / f'{stem}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')
    print(f'Saved: {png_path}')
    print(f'Saved: {pdf_path}')

## 4. Đọc annotation và kiểm tra dữ liệu

Mỗi CSV training tương ứng với một lớp. Cell ghép các CSV, tạo đường dẫn ảnh, đọc ground truth của official test set và in ra các thống kê cơ bản. Các con số được tính trực tiếp thay vì gõ tay.

In [ ]:
train_dir_candidates = list(DATA_ROOT.rglob('Final_Training/Images'))
test_dir_candidates = list(DATA_ROOT.rglob('Final_Test/Images'))
test_gt_candidates = list(DATA_ROOT.rglob('GT-final_test.csv'))

if not train_dir_candidates or not test_dir_candidates or not test_gt_candidates:
    raise FileNotFoundError('Không tìm thấy cấu trúc GTSRB sau khi giải nén.')

TRAIN_DIR = train_dir_candidates[0]
TEST_DIR = test_dir_candidates[0]
TEST_GT_PATH = test_gt_candidates[0]

train_csv_files = sorted(TRAIN_DIR.glob('*/*.csv'))
if len(train_csv_files) != 43:
    raise RuntimeError(f'Expected 43 training CSV files, found {len(train_csv_files)}.')

train_parts = []
for csv_path in train_csv_files:
    part = pd.read_csv(csv_path, sep=';')
    part['ImagePath'] = part['Filename'].map(lambda name: str(csv_path.parent / name))
    train_parts.append(part)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.read_csv(TEST_GT_PATH, sep=';')
test_df['ImagePath'] = test_df['Filename'].map(lambda name: str(TEST_DIR / name))

print('Training directory:', TRAIN_DIR)
print('Test directory:', TEST_DIR)
print('Training images:', len(train_df))
print('Official test images:', len(test_df))
print('Number of classes:', train_df['ClassId'].nunique())
print('Missing training files:', (~train_df['ImagePath'].map(os.path.exists)).sum())
print('Missing test files:', (~test_df['ImagePath'].map(os.path.exists)).sum())

display(train_df.head())

## 5. Figure 1 — Representative samples from all 43 classes

Đây nên là hình đầu tiên xuất hiện trong paper vì nó giới thiệu trực tiếp dữ liệu mà nhóm sử dụng. Mỗi ô là ROI thật được lấy từ một lớp GTSRB; không sử dụng ảnh minh họa bên ngoài.

In [ ]:
def read_rgb(path):
    image_bgr = cv2.imread(str(path))
    if image_bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

def crop_roi(image_rgb, row):
    x1 = int(row['Roi.X1'])
    y1 = int(row['Roi.Y1'])
    x2 = int(row['Roi.X2'])
    y2 = int(row['Roi.Y2'])
    return image_rgb[y1:y2 + 1, x1:x2 + 1]

representatives = (
    train_df.sort_values(['ClassId', 'Filename'])
    .groupby('ClassId', as_index=False)
    .first()
)

fig, axes = plt.subplots(7, 7, figsize=(11.5, 11.5))
for axis in axes.flat:
    axis.axis('off')

for position, row in representatives.iterrows():
    class_id = int(row['ClassId'])
    image = read_rgb(row['ImagePath'])
    roi = crop_roi(image, row)

    axis = axes.flat[position]
    axis.imshow(roi)
    short_name = textwrap.fill(CLASS_NAMES[class_id], width=18)
    axis.set_title(f'{class_id}: {short_name}', fontsize=6.3, pad=2)
    axis.axis('off')

fig.subplots_adjust(wspace=0.15, hspace=0.55)
save_figure(fig, 'fig_01_gtsrb_43_class_samples')
plt.show()

## 6. Figure 2 — Class distribution and original image dimensions

Biểu đồ bên trái chứng minh dữ liệu training mất cân bằng giữa các lớp. Biểu đồ bên phải mô tả kích thước ảnh gốc không đồng nhất, qua đó hỗ trợ lý do resize ảnh về 32 x 32 trước khi trích xuất HOG.

In [ ]:
class_counts = train_df['ClassId'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2), gridspec_kw={'width_ratios': [1.45, 1]})

axes[0].bar(class_counts.index, class_counts.values, color='#4C78A8', width=0.82)
axes[0].set_xlabel('Class ID')
axes[0].set_ylabel('Number of training images')
axes[0].set_xticks(range(0, 43, 2))
axes[0].grid(axis='y', alpha=0.25)
axes[0].grid(axis='x', visible=False)
axes[0].text(0.01, 0.96, '(a) Class distribution', transform=axes[0].transAxes, va='top', fontweight='bold')

hexbin = axes[1].hexbin(
    train_df['Width'],
    train_df['Height'],
    gridsize=35,
    mincnt=1,
    cmap='Blues',
    bins='log',
)
axes[1].set_xlabel('Original image width (pixels)')
axes[1].set_ylabel('Original image height (pixels)')
axes[1].grid(False)
axes[1].text(0.01, 0.96, '(b) Original image dimensions', transform=axes[1].transAxes, va='top', fontweight='bold')
colorbar = fig.colorbar(hexbin, ax=axes[1], pad=0.02)
colorbar.set_label('Log-scaled sample density')

fig.tight_layout()
save_figure(fig, 'fig_02_dataset_distribution_and_dimensions')
plt.show()

dataset_statistics = pd.DataFrame({
    'Statistic': [
        'Training images',
        'Official test images',
        'Number of classes',
        'Minimum class support',
        'Maximum class support',
        'Median class support',
    ],
    'Value': [
        len(train_df),
        len(test_df),
        train_df['ClassId'].nunique(),
        int(class_counts.min()),
        int(class_counts.max()),
        float(class_counts.median()),
    ],
})
dataset_statistics.to_csv(FIGURE_DIR / 'table_dataset_statistics.csv', index=False)
display(dataset_statistics)

## 7. Figure 3 — Actual preprocessing and feature-extraction pipeline

Hình này sử dụng một mẫu thật để minh họa lần lượt ảnh gốc với ROI annotation, ROI sau crop, ảnh 32 x 32, ảnh grayscale và HOG visualization. Khối cuối mô tả đúng StandardScaler và LinearSVC đang dùng trong notebook huấn luyện.

In [ ]:
stop_candidates = train_df[train_df['ClassId'] == 14].copy()
stop_candidates['RoiArea'] = (
    (stop_candidates['Roi.X2'] - stop_candidates['Roi.X1'] + 1)
    * (stop_candidates['Roi.Y2'] - stop_candidates['Roi.Y1'] + 1)
)
pipeline_row = stop_candidates.sort_values('RoiArea', ascending=False).iloc[0]

original = read_rgb(pipeline_row['ImagePath'])
cropped = crop_roi(original, pipeline_row)
resized = cv2.resize(cropped, IMG_SIZE, interpolation=cv2.INTER_AREA)
grayscale = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
hog_vector, hog_image = hog(
    grayscale,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
    feature_vector=True,
)
hog_display = exposure.rescale_intensity(hog_image, in_range=(0, max(hog_image.max(), 1e-8)))

fig, axes = plt.subplots(1, 6, figsize=(15, 3.0), gridspec_kw={'width_ratios': [1.15, 1, 1, 1, 1, 1.3]})

axes[0].imshow(original)
x1, y1 = int(pipeline_row['Roi.X1']), int(pipeline_row['Roi.Y1'])
x2, y2 = int(pipeline_row['Roi.X2']), int(pipeline_row['Roi.Y2'])
axes[0].add_patch(Rectangle((x1, y1), x2 - x1 + 1, y2 - y1 + 1, fill=False, edgecolor='#D62728', linewidth=1.7))
axes[0].set_title('Input image + ROI')

axes[1].imshow(cropped)
axes[1].set_title(f'Crop ROI\n{cropped.shape[1]} x {cropped.shape[0]}')

axes[2].imshow(resized)
axes[2].set_title('Resize\n32 x 32')

axes[3].imshow(grayscale, cmap='gray')
axes[3].set_title('Grayscale\n32 x 32')

axes[4].imshow(hog_display, cmap='gray')
axes[4].set_title(f'HOG descriptor\n{hog_vector.shape[0]}-D')

axes[5].axis('off')
axes[5].text(
    0.5, 0.5,
    'StandardScaler\n$\\downarrow$\nLinearSVC\n$C=1.0$\n43 classes',
    ha='center', va='center', fontsize=10,
    bbox={'boxstyle': 'round,pad=0.5', 'facecolor': '#F2F2F2', 'edgecolor': '#444444'},
)
axes[5].set_title('Classification')

for axis in axes[:5]:
    axis.axis('off')

fig.subplots_adjust(wspace=0.38)
fig.canvas.draw()
for left_axis, right_axis in zip(axes[:-1], axes[1:]):
    left_box = left_axis.get_position()
    right_box = right_axis.get_position()
    start = (left_box.x1 + 0.003, (left_box.y0 + left_box.y1) / 2)
    end = (right_box.x0 - 0.003, (right_box.y0 + right_box.y1) / 2)
    plt.annotate('', xy=end, xytext=start, xycoords='figure fraction', textcoords='figure fraction', arrowprops={'arrowstyle': '->', 'lw': 1.1})

save_figure(fig, 'fig_03_hog_linearsvc_pipeline')
plt.show()

print('Selected sample:', pipeline_row['ImagePath'])
print('ClassId:', int(pipeline_row['ClassId']))
print('HOG feature shape:', hog_vector.shape)

## 8. Trích xuất HOG cho toàn bộ training set

Cell này tái tạo đúng hàm của `traffic_sign.ipynb`: crop ROI, resize 32 x 32, grayscale, sau đó HOG với 9 orientations, cell 8 x 8, block 2 x 2 và L2-Hys. Feature được cache để lần chạy sau không phải trích xuất lại.

In [ ]:
def extract_hog_feature(image_path, roi):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    x1, y1, x2, y2 = roi
    cropped_image = image_rgb[y1:y2 + 1, x1:x2 + 1]
    resized_image = cv2.resize(cropped_image, IMG_SIZE, interpolation=cv2.INTER_AREA)
    gray_image = cv2.cvtColor(resized_image, cv2.COLOR_RGB2GRAY)

    feature = hog(
        gray_image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True,
    )
    return feature.astype('float32')

feature_cache_path = CACHE_DIR / 'gtsrb_hog_32x32_train_features.npz'

if feature_cache_path.exists():
    cached = np.load(feature_cache_path)
    X = cached['X']
    y = cached['y']
    print('Loaded cached training features.')
else:
    features = []
    labels = []

    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Extracting HOG'):
        roi = (
            int(row['Roi.X1']),
            int(row['Roi.Y1']),
            int(row['Roi.X2']),
            int(row['Roi.Y2']),
        )
        feature = extract_hog_feature(row['ImagePath'], roi)
        if feature is not None:
            features.append(feature)
            labels.append(int(row['ClassId']))

    X = np.asarray(features, dtype=np.float32)
    y = np.asarray(labels, dtype=np.int64)
    np.savez_compressed(feature_cache_path, X=X, y=y)
    print('Saved training feature cache.')

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Number of classes:', np.unique(y).size)

## 9. Huấn luyện và đánh giá trên validation split

Cell sử dụng đúng stratified 80/20 split, `random_state=42`, StandardScaler và LinearSVC của nhóm. Tất cả metric và bảng class-wise được tính từ dự đoán thật rồi lưu thành CSV.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        class_weight='balanced',
        max_iter=10000,
        dual=False,
        random_state=RANDOM_STATE,
    ),
)

training_start = time.perf_counter()
model.fit(X_train, y_train)
training_time_seconds = time.perf_counter() - training_start
y_val_pred = model.predict(X_val)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_val, y_val_pred, average='macro', zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_val, y_val_pred, average='weighted', zero_division=0
)

metrics = {
    'accuracy': float(accuracy_score(y_val, y_val_pred)),
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'weighted_precision': float(weighted_precision),
    'weighted_recall': float(weighted_recall),
    'weighted_f1': float(weighted_f1),
    'training_time_seconds_this_run': float(training_time_seconds),
    'num_train_samples': int(len(y_train)),
    'num_validation_samples': int(len(y_val)),
}

report = classification_report(y_val, y_val_pred, output_dict=True, zero_division=0)
per_class_rows = []
for class_id in range(43):
    values = report[str(class_id)]
    per_class_rows.append({
        'ClassId': class_id,
        'ClassName': CLASS_NAMES[class_id],
        'Precision': float(values['precision']),
        'Recall': float(values['recall']),
        'F1Score': float(values['f1-score']),
        'Support': int(values['support']),
    })
per_class_df = pd.DataFrame(per_class_rows)

with open(FIGURE_DIR / 'validation_metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)
per_class_df.to_csv(FIGURE_DIR / 'table_validation_per_class_metrics.csv', index=False)
joblib.dump(model, CACHE_DIR / 'hog_linearsvc_gtsrb.pkl')

display(pd.DataFrame({'Metric': metrics.keys(), 'Value': metrics.values()}))
display(per_class_df.sort_values('F1Score').head(10))

## 10. Figure 4 — Validation confusion matrix

Ma trận nhầm lẫn được chuẩn hóa theo từng lớp thật. Cách chuẩn hóa này phù hợp khi số lượng mẫu giữa các lớp khác nhau và giúp nhận ra các cặp lớp thường bị nhầm. Không ghi số trong từng ô vì ma trận 43 x 43 sẽ trở nên khó đọc.

In [ ]:
cm_normalized = confusion_matrix(
    y_val,
    y_val_pred,
    labels=np.arange(43),
    normalize='true',
)

fig, axis = plt.subplots(figsize=(9.2, 7.8))
sns.heatmap(
    cm_normalized,
    ax=axis,
    cmap='Blues',
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={'label': 'Proportion within true class'},
)
axis.set_xlabel('Predicted ClassId')
axis.set_ylabel('True ClassId')
axis.set_xticks(np.arange(0.5, 43.5, 2), labels=np.arange(0, 43, 2), rotation=0)
axis.set_yticks(np.arange(0.5, 43.5, 2), labels=np.arange(0, 43, 2), rotation=0)
fig.tight_layout()
save_figure(fig, 'fig_04_validation_confusion_matrix')
plt.show()

## 11. Figure 5 — Class-wise F1-score and validation support

Hình này liên kết trực tiếp hiệu năng từng lớp với số mẫu validation của lớp đó. Nó hỗ trợ phần Class-wise Discussion và giúp tránh kết luận thiếu căn cứ chỉ từ accuracy tổng thể.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13.5, 6.2), sharex=True, gridspec_kw={'height_ratios': [1.35, 1]})

axes[0].bar(per_class_df['ClassId'], per_class_df['F1Score'], color='#4C78A8', width=0.82)
axes[0].axhline(metrics['macro_f1'], color='#D62728', linestyle='--', linewidth=1.2, label=f"Macro F1 = {metrics['macro_f1']:.3f}")
axes[0].set_ylabel('Validation F1-score')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right', frameon=True)
axes[0].grid(axis='y', alpha=0.25)
axes[0].grid(axis='x', visible=False)

axes[1].bar(per_class_df['ClassId'], per_class_df['Support'], color='#9ECAE1', width=0.82)
axes[1].set_xlabel('Class ID')
axes[1].set_ylabel('Validation support')
axes[1].set_xticks(range(43))
axes[1].grid(axis='y', alpha=0.25)
axes[1].grid(axis='x', visible=False)

fig.tight_layout()
save_figure(fig, 'fig_05_classwise_f1_and_support')
plt.show()

## 12. Figure 6 — Qualitative predictions on official test images

Cell chọn 12 ảnh test bằng random seed cố định, crop theo official ROI annotation và hiển thị true label cùng predicted label. Đây chỉ là đánh giá định tính; hình không được dùng để thay thế full test-set accuracy.

In [ ]:
sample_test_df = test_df.sample(n=12, random_state=RANDOM_STATE).reset_index(drop=True)
prediction_rows = []

fig, axes = plt.subplots(3, 4, figsize=(10.8, 8.4))
for axis, (_, row) in zip(axes.flat, sample_test_df.iterrows()):
    roi_coordinates = (
        int(row['Roi.X1']),
        int(row['Roi.Y1']),
        int(row['Roi.X2']),
        int(row['Roi.Y2']),
    )
    feature = extract_hog_feature(row['ImagePath'], roi_coordinates)
    predicted_id = int(model.predict(feature.reshape(1, -1))[0])
    true_id = int(row['ClassId'])
    correct = predicted_id == true_id

    image = read_rgb(row['ImagePath'])
    roi_image = crop_roi(image, row)
    axis.imshow(roi_image)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(
        f'True: {true_id} | Pred: {predicted_id}',
        fontsize=8,
        color='#1B7837' if correct else '#B2182B',
    )
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_edgecolor('#1B7837' if correct else '#B2182B')

    prediction_rows.append({
        'Filename': row['Filename'],
        'TrueClassId': true_id,
        'TrueClassName': CLASS_NAMES[true_id],
        'PredictedClassId': predicted_id,
        'PredictedClassName': CLASS_NAMES[predicted_id],
        'Correct': correct,
    })

fig.tight_layout()
save_figure(fig, 'fig_06_official_test_sample_predictions')
plt.show()

sample_predictions_df = pd.DataFrame(prediction_rows)
sample_predictions_df.to_csv(FIGURE_DIR / 'table_official_test_sample_predictions.csv', index=False)
display(sample_predictions_df)

## 13. Đóng gói kết quả để tải về

Cell cuối nén toàn bộ PNG, PDF, CSV và JSON. Chỉ chèn vào paper các hình đã được kiểm tra sau khi chạy notebook thành công.

In [ ]:
import shutil

archive_path = shutil.make_archive('/content/gtsrb_paper_figures', 'zip', FIGURE_DIR)
print('Created:', archive_path)
print('Generated files:')
for path in sorted(FIGURE_DIR.iterdir()):
    print('-', path.name)

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print('Không chạy trong Colab; hãy tải file zip từ đường dẫn ở trên.')